# How the VAE sees a cat photo

The ready-made Stable Diffusion autoencoder compresses a 512x512x3 image into a 64x64x4 latent and reconstructs it back.
Diffusion will work only with latents, so whatever the VAE loses here, the model will not be able to draw later.
The notebook shows what exactly is lost on a cat photo and which resolution to use.

In [ ]:
import os
import sys
from pathlib import Path

# work from the repository root: imports from terra_diffusion and relative paths like runs/ then work
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'terra_diffusion').is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import AutoencoderKL

from terra_diffusion.images import center_crops, load_images

VAE_NAME = 'stabilityai/sd-vae-ft-mse'    # weights are already in ~/.cache/huggingface
DATA_DIR = 'dataset_unique'
SIZE = 512                                # side of the square center crop
BATCH = 16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

vae = AutoencoderKL.from_pretrained(VAE_NAME, local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)

SCALE = vae.config.scaling_factor
F_DOWN = 2 ** (len(vae.config.block_out_channels) - 1)
n_par = sum(p.numel() for p in vae.parameters())
print(f'{VAE_NAME}: {n_par / 1e6:.1f} M parameters, {F_DOWN}x compression, '
      f'{vae.config.latent_channels} channels, scaling_factor {SCALE}, device {DEVICE}')

In [ ]:
@torch.no_grad()
def encode(x):
    """(B, 3, S, S) in [-1, 1] -> mean and log-variance of the latent, both (B, 4, S/8, S/8)."""
    d = vae.encode(x.to(DEVICE)).latent_dist
    return d.mean, d.logvar


@torch.no_grad()
def decode(z):
    """Latent without multiplying by scaling_factor -> image in [-1, 1]."""
    return vae.decode(z).sample.clamp(-1, 1)


def to_uint8(x):
    """[-1, 1] -> uint8 on CPU, so 395 images take 300 MB, not 6 GB."""
    return ((x + 1) * 127.5).round().clamp(0, 255).byte().cpu()


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) for imshow."""
    return img.permute(1, 2, 0).numpy()


def psnr(a, b):
    """PSNR in dB for each image, a and b: (B, 3, H, W) uint8. Higher = better."""
    mse = ((a.float() - b.float()) ** 2).mean((1, 2, 3))
    return 10 * torch.log10(255 ** 2 / mse)

## All photos through the VAE

We take the latent mean, without the random addition: this shows the best result the VAE is capable of.

In [ ]:
t0 = time.time()
files, imgs = load_images(DATA_DIR, SIZE)
print(f'photos {len(imgs)}, loading {time.time() - t0:.1f} s')

t0 = time.time()
X, R, Z, S = [], [], [], []
for i in range(0, len(imgs), BATCH):
    x = center_crops(imgs[i:i + BATCH], SIZE)
    mean, logvar = encode(x)
    X.append(to_uint8(x))
    R.append(to_uint8(decode(mean)))
    Z.append(mean.cpu())
    S.append(torch.exp(0.5 * logvar).cpu())
X, R, Z, S = torch.cat(X), torch.cat(R), torch.cat(Z), torch.cat(S)
P = psnr(X, R).numpy()

print(f'round trip in {time.time() - t0:.1f} s, {(time.time() - t0) / len(imgs) * 1e3:.0f} ms per photo')
print(f'PSNR: median {np.median(P):.1f} dB, 10% quantile {np.quantile(P, 0.1):.1f}, '
      f'min {P.min():.1f}, max {P.max():.1f}')

In [ ]:
# Diffusion needs latents with variance around 1, so we check the multiplier on our own photos
mean_c = Z.mean((0, 2, 3))
std_c = Z.std((0, 2, 3))
print('chan       mean       std')
for c in range(Z.shape[1]):
    print(f'{c:5d} {mean_c[c].item():10.3f} {std_c[c].item():9.3f}')

print(f'\nstd over all channels {Z.std().item():.3f}, so the multiplier for our photos is {1 / Z.std().item():.5f} '
      f'(model has {SCALE})')
print(f'latent own noise: mean sigma {S.mean().item():.4f}, '
      f'this is {S.mean().item() / Z.std().item() * 100:.1f}% of the latent spread')

## Worst, average and best photos

Rows: original, VAE result, error amplified 4 times.
PSNR is computed over the whole frame, so the worst photos are those with fine background texture, not with a badly reconstructed cat.

In [ ]:
order = np.argsort(P)
mid = len(order) // 2
show = [*order[:2], *order[mid - 1:mid + 1], *order[-2:]]

fig, axs = plt.subplots(3, len(show), figsize=(3.2 * len(show), 9.8))
for j, k in enumerate(show):
    err = (X[k].float() - R[k].float()).abs().mean(0).numpy() / 255
    axs[0, j].imshow(to_np(X[k]))
    axs[1, j].imshow(to_np(R[k]))
    axs[2, j].imshow((err * 4).clip(0, 1), cmap='magma', vmin=0, vmax=1)
    axs[0, j].set_title(f'#{k}, PSNR {P[k]:.1f} dB')
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Overview

The numbers above the images are needed for the `ZOOMS` list in the next cell. Lines are drawn every 128 pixels.

In [ ]:
START = 0
N_SHOW = 48
COLS = 8

idx = list(range(START, min(START + N_SHOW, len(X))))
rows = (len(idx) + COLS - 1) // COLS
fig, axs = plt.subplots(rows, COLS, figsize=(2.6 * COLS, 2.8 * rows))
for ax in axs.flat:
    ax.axis('off')
for ax, k in zip(axs.flat, idx):
    ax.axis('on')
    ax.imshow(to_np(X[k]))
    ax.set_title(f'№{k}', fontsize=10)
    ax.set_xticks([128, 256, 384], labels=[])
    ax.set_yticks([128, 256, 384], labels=[])
    ax.grid(color='w', alpha=0.4, lw=0.5)
    ax.tick_params(length=0)
plt.tight_layout()
plt.show()

## Face and fur close up

`ZOOMS` holds patches (photo number, y, x), the center coordinates in pixels of the 512x512 image.
Rows: original, VAE result, error amplified 4 times.

In [ ]:
ZOOMS = [(0, 200, 190), (1, 185, 220), (8, 150, 218), (12, 150, 268), (28, 205, 300), (37, 163, 235), (0, 190, 420), (374, 100, 100)]
WIN = 128     # side of the patch in pixels


def crop(img, y, x, win):
    """A win x win square centered at (y, x), shifted so as not to go past the edge."""
    _, H, W = img.shape
    top = min(max(y - win // 2, 0), H - win)
    left = min(max(x - win // 2, 0), W - win)
    return img[:, top:top + win, left:left + win]


fig, axs = plt.subplots(3, len(ZOOMS), figsize=(3.2 * len(ZOOMS), 9.8))
for j, (k, y, x) in enumerate(ZOOMS):
    a, b = crop(X[k], y, x, WIN), crop(R[k], y, x, WIN)
    err = (a.float() - b.float()).abs().mean(0).numpy() / 255
    axs[0, j].imshow(to_np(a), interpolation='nearest')
    axs[1, j].imshow(to_np(b), interpolation='nearest')
    axs[2, j].imshow((err * 4).clip(0, 1), cmap='magma', vmin=0, vmax=1, interpolation='nearest')
    axs[0, j].set_title(f'#{k}, patch {WIN}x{WIN}')
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## What is in the latent

The same 512x512, but through the eyes of diffusion: 4 maps of 64x64. The color image is still recognizable in the latent,
so the U-Net works with spatial convolutions here too.

In [ ]:
IDX = 36

z = Z[IDX] * SCALE     # the scale at which diffusion will see the latent
lim = z.abs().quantile(0.995).item()

fig, axs = plt.subplots(1, 6, figsize=(21, 3.8))
axs[0].imshow(to_np(X[IDX]))
axs[0].set_title('original, 512x512x3')
for c in range(z.shape[0]):
    axs[c + 1].imshow(z[c].numpy(), cmap='gray', vmin=-lim, vmax=lim, interpolation='nearest')
    axs[c + 1].set_title(f'channel {c}, 64x64')
axs[5].imshow(to_np(R[IDX]))
axs[5].set_title(f'decoder, PSNR {P[IDX]:.1f} dB')
for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Which resolution to choose

The same photos at different sizes, only those whose short side is at least 1024: they do not have to be stretched.
PSNR cannot be compared across sizes: at a smaller size the same details are packed more densely, and it is a different task.
Look at the patches in the next figure: what remains of the face at latent 32x32 (256) and 64x64 (512).

In [ ]:
SIZES = [256, 512, 768, 1024]
N_TEST = 32

short = np.array([min(Image.open(p).size) for p in files])
big = np.flatnonzero(short >= 1024)
test = set(big[np.linspace(0, len(big) - 1, N_TEST).round().astype(int)].tolist())
zooms_big = [z for z in ZOOMS if short[z[0]] >= 1024]
test |= {z[0] for z in zooms_big}
print(f'short side >= 1024 for {len(big)} photos of {len(files)}, {len(test)} in the test')

names = {files[k].name for k in test}
skip = {f.name for f in files} - names

res = {}
print('size     latent       PSNR, median   ms per photo')
for size in SIZES:
    fs, ims = load_images(DATA_DIR, size, exclude=skip)
    bs = max(2, int(BATCH * (512 / size) ** 2))
    t0 = time.time()
    xs, rs = [], []
    for i in range(0, len(ims), bs):
        x = center_crops(ims[i:i + bs], size)
        xs.append(to_uint8(x))
        rs.append(to_uint8(decode(encode(x)[0])))
    dt = time.time() - t0
    xs, rs = torch.cat(xs), torch.cat(rs)
    res[size] = {'x': dict(zip([f.name for f in fs], xs)), 'r': dict(zip([f.name for f in fs], rs))}
    lat = size // F_DOWN
    print(f'{size:6d}   {lat:3d}x{lat:<3d}x4 {np.median(psnr(xs, rs).numpy()):12.1f} dB {dt / len(fs) * 1e3:10.0f}')
    torch.cuda.empty_cache()

In [ ]:
# The same patch: original at 1024 on the left, then the VAE result at each size
rows = len(zooms_big)
fig, axs = plt.subplots(rows, len(SIZES) + 1, figsize=(3.4 * (len(SIZES) + 1), 3.5 * rows), squeeze=False)
for i, (k, y, x) in enumerate(zooms_big):
    name = files[k].name
    s0 = SIZES[-1] / SIZE
    axs[i, 0].imshow(to_np(crop(res[SIZES[-1]]['x'][name], int(y * s0), int(x * s0), int(WIN * s0))),
                     interpolation='nearest')
    axs[i, 0].set_title(f'#{k}, original {SIZES[-1]}')
    for j, size in enumerate(SIZES):
        s = size / SIZE
        axs[i, j + 1].imshow(to_np(crop(res[size]['r'][name], int(y * s), int(x * s), int(WIN * s))),
                             interpolation='nearest')
        axs[i, j + 1].set_title(f'VAE, {size}')
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()